# 第12章 利率互换 — 编程实验完整解答

[![Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/albertandking/fixed-income/blob/main/notebooks/solutions/ch12_solutions.ipynb) [![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/albertandking/fixed-income/main?labpath=notebooks/solutions/ch12_solutions.ipynb)

本 notebook 给出本章全部编程实验的完整可运行解答；联网（akshare）部分以注释/降级方式给出，离线也能跑通。


In [ ]:
# 自举单元：Colab/Binder 自动安装 fi；本地跳过。
import importlib.util, sys, subprocess
if importlib.util.find_spec('fi') is None:
    if 'google.colab' in sys.modules:
        subprocess.run(['git','clone','--depth','1','https://github.com/albertandking/fixed-income.git','/content/fi-book'],check=False)
        subprocess.run([sys.executable,'-m','pip','install','-e','/content/fi-book'],check=False)
    else:
        print('提示：仓库根目录执行 `uv sync --extra all` 后运行本 notebook。')


## 编程实验 7：bootstrap 互换曲线 + 复原验证


In [ ]:
from fi import swap as sw
par = [0.024,0.026,0.028,0.030,0.031]
z, df = sw.bootstrap_swap_curve(par)
for n,(p,zi,d) in enumerate(zip(par,z,df), start=1): print(f'{n}y: par={p*100:.1f}% 即期={zi*100:.4f}% DF={d:.6f}')
print('用DF重算5y平价互换 =', round(sw.par_swap_rate(df,[1]*5)*100,4), '% (复原)')


## 编程实验 8：payer 互换盯市价值（图12-1）


In [ ]:
import numpy as np
from fi import plotting
plotting.use_chinese_style()
rates = np.linspace(0.01,0.05,41); taus=[1]*5
vals = [sw.swap_value(0.025, [(1+z)**-t for t in range(1,6)], taus, 1e8, True)/1e4 for z in rates]
fig, ax = plotting.new_axes()
ax.plot(rates*100, vals, label='payer互换(付2.5%)'); ax.axhline(0, ls=':', color='gray'); ax.axvline(2.5, ls=':', color='gray')
ax.set_xlabel('市场利率 (%)'); ax.set_ylabel('盯市价值(万元)'); ax.set_title('payer利率升则获利'); ax.legend(); fig.tight_layout()


## 编程实验 9：fi vs QuantLib VanillaSwap


In [ ]:
dfs = [(1.03)**-t for t in range(1,6)]; taus=[1]*5
v = sw.swap_value(0.025, dfs, taus, 1e8, True)
import QuantLib as ql
today = ql.Date(15,6,2026); ql.Settings.instance().evaluationDate = today; dc, cal = ql.Actual365Fixed(), ql.NullCalendar()
disc = ql.YieldTermStructureHandle(ql.FlatForward(today,0.03,dc))
idx = ql.IborIndex('Idx', ql.Period(1,ql.Years), 0, ql.CNYCurrency(), cal, ql.Unadjusted, False, dc, disc)
sched = ql.Schedule(today, today+ql.Period(5,ql.Years), ql.Period(1,ql.Years), cal, ql.Unadjusted, ql.Unadjusted, ql.DateGeneration.Forward, False)
qs = ql.VanillaSwap(ql.VanillaSwap.Payer, 1e8, sched, 0.025, dc, sched, idx, 0.0, dc); qs.setPricingEngine(ql.DiscountingSwapEngine(disc))
print(f'fi NPV={v:,.0f} par=3.0000%   QuantLib NPV={qs.NPV():,.0f} par={qs.fairRate()*100:.4f}%')
print('差异来自计息惯例/真实日历')
